In [4]:
!python -m pip install -U python-woc pandas matplotlib
%pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.4/6.4 MB 15.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 14.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 16.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.4/199.4 kB 15.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 13.0 MB/s eta 0:00:0000:0100:01
  DEPRECATION: python-lzf is being installed using the legacy 'setup.py install' method, because it does not have a 'pyproject.toml' and the 'wheel' package is not installed. pip 23.1 will enforce this behaviour change. A possible replacement is to enable the '--use-pep517' option. Discussion can be found at https://github.com/pypa/pip/issues/8559
  Running setu

In [1]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
projects = [
    'jlizier/jidt',
    'tencent/turbotransformers',
    'adda-team/adda',
    'envirocar/envirocar-app',
    'juliamath/gsl.jl',
    'snakemake/snakemake',
    'cpmech/gosl',
    'open-quantum-safe/ci-containers',
    'kul-forbes/forbes',
    'poissonconsulting/tmbr'
]
list_df_commits = []
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(1)

,sha1,project
0,001227c730f02caaa878e6e36e076de7c0ca9a9a,jlizier_jidt


In [2]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

  4%|███▎                                                                           | 128/3023 [02:13<50:24,  1.04s/it]

Got Errors {'bb81f52f1e92666671467a04870e0c514d871d4e': 'Key bb81f52f1e92666671467a04870e0c514d871d4e not found in /da5_fast/All.sha1c/commit_59.tch'}


 11%|████████▉                                                                      | 342/3023 [05:57<46:39,  1.04s/it]

Got Errors {'0efedec1853d85f9816c60d76e28dd239973fa23': 'Key 0efedec1853d85f9816c60d76e28dd239973fa23 not found in /da5_fast/All.sha1c/commit_14.tch'}


 12%|█████████                                                                      | 349/3023 [06:04<46:34,  1.04s/it]

...

Got Errors {'d334d724290100db66f03de331950f2403c29bbe': 'Key d334d724290100db66f03de331950f2403c29bbe not found in /da5_fast/All.sha1c/commit_83.tch'}


 91%|███████████████████████████████████████████████████████████████████████▎      | 2763/3023 [48:11<04:29,  1.04s/it]

Got Errors {'8030172beedbdc2199c77dd4f6de64c9ddf6f2e5': 'Key 8030172beedbdc2199c77dd4f6de64c9ddf6f2e5 not found in /da5_fast/All.sha1c/commit_0.tch'}


100%|██████████████████████████████████████████████████████████████████████████████| 3023/3023 [52:42<00:00,  1.05s/it]


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,001227c730f02caaa878e6e36e076de7c0ca9a9a,556ce9723fe15474c2bbb88426b0d08c9cbe9468,[118e32d9ad53439fdb061975f121cf992dcf6c37],Pedro Martinez Mediano <pedro.martinez-mediano...,1495429412,+1000,Pedro Martinez Mediano <pedro.martinez-mediano...,1495429412,+1000,Added Java support for GPU surrogate calculati...,001227c730f02caaa878e6e36e076de7c0ca9a9a,jlizier_jidt
1,001fb764cfac4ee4d374269c91599ca3908c0b7f,cda935e51d935ce4538b76184f2b22ccb9c36232,[13e6d2964063b543722b667a64dc772868fa962f],joseph.lizier@gmail.com <joseph.lizier@gmail.c...,1407242645,+0000,joseph.lizier@gmail.com <joseph.lizier@gmail.c...,1407242645,+0000,Refactoring discrete TE calculator to take sou...,001fb764cfac4ee4d374269c91599ca3908c0b7f,jlizier_jidt


In [3]:
# what does the first record looks like?
# commit  parents
v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
print(v)

{'commit': '001227c730f02caaa878e6e36e076de7c0ca9a9a', 'tree': '556ce9723fe15474c2bbb88426b0d08c9cbe9468', 'parent': ['118e32d9ad53439fdb061975f121cf992dcf6c37'], 'author': 'Pedro Martinez Mediano <pedro.martinez-mediano13@imperial.ac.uk>', 'author_time': 1495429412, 'author_tz': '+1000', 'committer': 'Pedro Martinez Mediano <pedro.martinez-mediano13@imperial.ac.uk>', 'committer_time': 1495429412, 'committer_tz': '+1000', 'message': 'Added Java support for GPU surrogate calculation.\n'}


In [4]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
dfinf = pd.DataFrame(columns=['project', 'commit', 'author', 'time', 'message'])
for k in commit_data:
  row = pd.Series({'project':prj, 'commit': k['commit'], 'author': k['author'], 'time': k['author_time'], 'message':k['message']})
  dfinf = pd.concat([dfinf, row.to_frame().T ], ignore_index=True)


In [5]:
# check if it has the right content
dfinf.head(1)

,project,commit,author,time,message
0,poissonconsulting_tmbr,001227c730f02caaa878e6e36e076de7c0ca9a9a,Pedro Martinez Mediano <pedro.martinez-mediano...,1495429412,Added Java support for GPU surrogate calculati...


In [6]:
#mode a means append, so you have all your projects in the same file
yournetid='dkim68'
dfinf.to_csv(yournetid+'_project_summary.csv', index=False,sep=';', mode='a', header=False)

# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github

In [13]:
import requests
import pandas as pd
from datetime import datetime

projects = [
    'jlizier/jidt',
    'tencent/turbotransformers',
    'adda-team/adda',
    'envirocar/envirocar-app',
    'juliamath/gsl.jl',
    'snakemake/snakemake',
    'cpmech/gosl',
    'open-quantum-safe/ci-containers',
    'kul-forbes/forbes',
    'poissonconsulting/tmbr'
]

rows = []

for repo in projects:
    repo_data = requests.get(
        f"https://api.github.com/repos/{repo}"
    ).json()

    commits = requests.get(
        f"https://api.github.com/repos/{repo}/commits",
        params={"per_page": 1}
    ).json()

    if commits and isinstance(commits, list):
        last_commit_date = commits[0]["commit"]["committer"]["date"]
    else:
        last_commit_date = None

    rows.append({
        "repo": repo,
        "stars": repo_data.get("stargazers_count"),
        "forks": repo_data.get("forks_count"),
        "last_commit_date": last_commit_date
    })

github_summary = pd.DataFrame(rows)
github_summary

,repo,stars,forks,last_commit_date
0,jlizier/jidt,306,83,2026-08-03T02:27:32Z
1,tencent/turbotransformers,1551,208,2025-07-18T03:51:17Z
2,adda-team/adda,132,65,2026-03-30T08:45:28Z
3,envirocar/envirocar-app,88,158,2026-02-10T12:09:53Z
4,juliamath/gsl.jl,101,30,2026-06-22T14:28:28Z
5,snakemake/snakemake,2878,656,2026-09-11T14:03:03Z
6,cpmech/gosl,1880,148,2025-12-30T00:20:17Z
7,open-quantum-safe/ci-containers,5,13,2026-07-26T06:03:25Z
8,kul-forbes/forbes,27,11,2025-01-27T13:40:34Z
9,poissonconsulting/tmbr,0,1,2026-09-01T07:20:55Z


In [14]:
woc_summary = (
    df_commit_data
    .groupby('project')
    .agg(
        num_commits=('commit', 'count'),
        num_authors=('author', 'nunique'),
        min_time=('author_time', 'min'),
        max_time=('author_time', 'max')
    )
    .reset_index()
)

woc_summary['min_time'] = pd.to_datetime(woc_summary['min_time'], unit='s')
woc_summary['max_time'] = pd.to_datetime(woc_summary['max_time'], unit='s')

woc_summary

,project,num_commits,num_authors,min_time,max_time
0,adda-team_adda,2200,48,2005-04-07 09:24:16,2025-10-15 12:41:27
1,cpmech_gosl,2838,27,2015-02-09 23:00:38,2025-08-07 21:33:55
2,envirocar_envirocar-app,3811,105,2013-04-10 08:53:02,2026-03-03 10:56:29
3,jlizier_jidt,1741,27,2012-05-07 23:25:25,2025-11-05 04:14:00
4,juliamath_gsl.jl,459,51,2013-03-27 05:40:57,2025-08-14 15:06:31
5,kul-forbes_forbes,442,12,2015-04-24 17:36:12,2019-01-18 09:27:43
6,open-quantum-safe_ci-containers,391,32,2018-10-17 17:40:20,2025-10-25 06:51:44
7,poissonconsulting_tmbr,355,24,2016-09-20 19:57:04,2025-06-03 22:41:20
8,snakemake_snakemake,16416,804,2011-11-13 22:33:28,2026-04-18 11:32:12
9,tencent_turbotransformers,1542,45,2019-10-28 03:29:53,2025-07-18 03:51:17


In [15]:
github_summary['project'] = (
    github_summary['repo']
    .str.lower()
    .str.replace('/', '_', regex=False)
)

In [16]:
project_summary = github_summary.merge(
    woc_summary,
    on='project',
    how='left'
)

project_summary

,repo,stars,forks,last_commit_date,project,num_commits,num_authors,min_time,max_time
0,jlizier/jidt,306,83,2026-08-03T02:27:32Z,jlizier_jidt,1741,27,2012-05-07 23:25:25,2025-11-05 04:14:00
1,tencent/turbotransformers,1551,208,2025-07-18T03:51:17Z,tencent_turbotransformers,1542,45,2019-10-28 03:29:53,2025-07-18 03:51:17
2,adda-team/adda,132,65,2026-03-30T08:45:28Z,adda-team_adda,2200,48,2005-04-07 09:24:16,2025-10-15 12:41:27
3,envirocar/envirocar-app,88,158,2026-02-10T12:09:53Z,envirocar_envirocar-app,3811,105,2013-04-10 08:53:02,2026-03-03 10:56:29
4,juliamath/gsl.jl,101,30,2026-06-22T14:28:28Z,juliamath_gsl.jl,459,51,2013-03-27 05:40:57,2025-08-14 15:06:31
5,snakemake/snakemake,2878,656,2026-09-11T14:03:03Z,snakemake_snakemake,16416,804,2011-11-13 22:33:28,2026-04-18 11:32:12
6,cpmech/gosl,1880,148,2025-12-30T00:20:17Z,cpmech_gosl,2838,27,2015-02-09 23:00:38,2025-08-07 21:33:55
7,open-quantum-safe/ci-containers,5,13,2026-07-26T06:03:25Z,open-quantum-safe_ci-containers,391,32,2018-10-17 17:40:20,2025-10-25 06:51:44
8,kul-forbes/forbes,27,11,2025-01-27T13:40:34Z,kul-forbes_forbes,442,12,2015-04-24 17:36:12,2019-01-18 09:27:43
9,poissonconsulting/tmbr,0,1,2026-09-01T07:20:55Z,poissonconsulting_tmbr,355,24,2016-09-20 19:57:04,2025-06-03 22:41:20
